### Procesamiento de Lenguaje Natural I
# Desafío 1


In [1]:
%pip install numpy scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/gabybohorquez/.local/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/gabybohorquez/.local/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/gabybohorquez/.local/li

AttributeError: _ARRAY_API not found

Utilizamos 20newsgroups por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palabra que no está en el documento.

In [10]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## Consigna del Desafío 1
Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.



1. Vectorizar documentos
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

2. Construir un modelo de clasificación por prototipos (tipo zero-shot).
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

NO cambiar el hiperparámetro ngram_range de los vectorizadores.

4. Transponer la matriz documento-término.
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables.


---
## Resolución

### 1. Vectorización de documentos y análisis de similaridad

In [24]:
# Agarramos 5 documentos random (con seed fija para que sea reproducible)
np.random.seed(42)
random_indices = np.random.choice(len(newsgroups_train.data), size=5, replace=False)
print("Índices seleccionados:", random_indices)

Índices seleccionados: [7492 3546 5582 4793 3813]


In [25]:
# Para cada doc, calculamos similaridad coseno contra todos los demás
# con todos los documentos de entrenamiento y mostramos los 5 más similares
for doc_idx in random_indices:
    print(f"\n{'='*80}")
    print(f"Documento #{doc_idx}")
    print(f"Clase: {newsgroups_train.target_names[y_train[doc_idx]]}")
    print(f"Texto (primeros 200 caracteres): {newsgroups_train.data[doc_idx][:200]}...")
    print(f"{'='*80}")

    # Similaridad coseno contra todos
    similarities = cosine_similarity(X_train[doc_idx], X_train)[0]

    # Top 5 más similares (sin contarse a sí mismo)
    most_similar_indices = np.argsort(similarities)[::-1][1:6]

    print("\nTop 5 documentos más similares:")
    for rank, sim_idx in enumerate(most_similar_indices, 1):
        sim_class = newsgroups_train.target_names[y_train[sim_idx]]
        sim_score = similarities[sim_idx]
        print(f"  {rank}. Doc #{sim_idx} | Clase: {sim_class} | Similaridad: {sim_score:.4f}")
        print(f"     Texto: {newsgroups_train.data[sim_idx][:100]}...")


Documento #7492
Clase: comp.sys.mac.hardware
Texto (primeros 200 caracteres): Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging o...

Top 5 documentos más similares:
  1. Doc #10935 | Clase: comp.sys.mac.hardware | Similaridad: 0.6665
     Texto: Hey everybody:

   I want to buy a mac and I want to get a good price...who doesn't?  So,
could anyo...
  2. Doc #7258 | Clase: comp.sys.ibm.pc.hardware | Similaridad: 0.3476
     Texto: Hay all:

    Has anyone out there heard of any performance stats on the fabled p24t.
 I was wonderi...
  3. Doc #4971 | Clase: comp.sys.mac.hardware | Similaridad: 0.1799
     Texto: Could someone please send instructions for installing simms and vram to 
jmk13@po.cwru.edu?  He's ju...
  4. Doc #4303 | Clase: misc.forsale | Similaridad: 0.1547
     Texto: For sale:

Roland D-50: $700 or best offer.
Excellent con

Se ve que la similaridad coseno funciona bien para agrupar documentos del mismo tema. Si dos documentos usan palabras parecidas, sus vectores TF-IDF van a quedar cerca. En algunos casos aparece un documento de otra clase, pero generalmente son temas relacionados, como `sci.space` y `sci.electronics` que comparten bastante vocabulario técnico.

### 2. Clasificación por prototipos (zero-shot)

In [26]:
# Vectorizamos test con el mismo vectorizador que fiteamos en train
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

In [27]:
# Clasificación por prototipos (tipo zero-shot):
# La idea: para cada doc de test, buscamos el doc de train más parecido
# y le copiamos la etiqueta
y_pred_prototype = []

# Calculamos similaridad de test vs train
# Lo hacemos en batches para no volar la memoria
batch_size = 500
for i in range(0, X_test.shape[0], batch_size):
    batch_end = min(i + batch_size, X_test.shape[0])
    # Similaridad coseno del batch contra todo train
    sim_matrix = cosine_similarity(X_test[i:batch_end], X_train)
    # Nos quedamos con el más similar de cada uno
    most_similar = np.argmax(sim_matrix, axis=1)
    # Le ponemos la etiqueta del doc de train más parecido
    y_pred_prototype.extend(y_train[most_similar])

y_pred_prototype = np.array(y_pred_prototype)

In [28]:
# Veamos qué tal anduvo
f1_proto = f1_score(y_test, y_pred_prototype, average='macro')
print(f"F1-Score Macro (clasificación por prototipos): {f1_proto:.4f}")

F1-Score Macro (clasificación por prototipos): 0.5050


Lo que hace es simple: comparar el documento de test contra todos los de train y asignarle la etiqueta del más parecido. No se entrena nada, solo comparamos vectores. El F1 que da no está mal considerando lo básico que es, pero un modelo entrenado le gana sin mucho esfuerzo.

### 3. Optimización de modelos Naïve Bayes

Ahora vamos a probar distintas combinaciones de vectorizadores y modelos para maximizar el F1-Score Macro.

Ojo: NO tocamos el `ngram_range`, como dice la consigna.

In [29]:
from sklearn.metrics import classification_report

# Vamos a probar varias configs
results = []

# --- Exp 1: TF-IDF default + MultinomialNB ---
vect_1 = TfidfVectorizer()
X_train_1 = vect_1.fit_transform(newsgroups_train.data)
X_test_1 = vect_1.transform(newsgroups_test.data)

clf_1 = MultinomialNB()
clf_1.fit(X_train_1, y_train)
y_pred_1 = clf_1.predict(X_test_1)
f1_1 = f1_score(y_test, y_pred_1, average='macro')
results.append(('TF-IDF default + MultinomialNB', f1_1))
print(f"Exp 1 - TF-IDF default + MultinomialNB: F1={f1_1:.4f}")

Exp 1 - TF-IDF default + MultinomialNB: F1=0.5854


In [30]:
# --- Exp 2: TF-IDF default + ComplementNB ---
# ComplementNB fue pensado para datasets con clases desbalanceadas
clf_2 = ComplementNB()
clf_2.fit(X_train_1, y_train)
y_pred_2 = clf_2.predict(X_test_1)
f1_2 = f1_score(y_test, y_pred_2, average='macro')
results.append(('TF-IDF default + ComplementNB', f1_2))
print(f"Exp 2 - TF-IDF default + ComplementNB: F1={f1_2:.4f}")

Exp 2 - TF-IDF default + ComplementNB: F1=0.6930


In [31]:
# --- Exp 3: TF-IDF con sublinear_tf + ComplementNB ---
# sublinear_tf aplica 1 + log(tf) en vez del tf puro,
# así las palabras super frecuentes no dominan tanto
vect_3 = TfidfVectorizer(sublinear_tf=True)
X_train_3 = vect_3.fit_transform(newsgroups_train.data)
X_test_3 = vect_3.transform(newsgroups_test.data)

clf_3 = ComplementNB()
clf_3.fit(X_train_3, y_train)
y_pred_3 = clf_3.predict(X_test_3)
f1_3 = f1_score(y_test, y_pred_3, average='macro')
results.append(('TF-IDF sublinear + ComplementNB', f1_3))
print(f"Exp 3 - TF-IDF sublinear + ComplementNB: F1={f1_3:.4f}")

Exp 3 - TF-IDF sublinear + ComplementNB: F1=0.6920


In [32]:
# --- Exp 4: TF-IDF con max_df y min_df + ComplementNB ---
# max_df: sacamos palabras que aparecen en >90% de docs (tipo stopwords)
# min_df: sacamos las que aparecen en <5 docs (ruido)
vect_4 = TfidfVectorizer(sublinear_tf=True, max_df=0.9, min_df=5)
X_train_4 = vect_4.fit_transform(newsgroups_train.data)
X_test_4 = vect_4.transform(newsgroups_test.data)

clf_4 = ComplementNB()
clf_4.fit(X_train_4, y_train)
y_pred_4 = clf_4.predict(X_test_4)
f1_4 = f1_score(y_test, y_pred_4, average='macro')
results.append(('TF-IDF sublinear+max_df+min_df + ComplementNB', f1_4))
print(f"Exp 4 - TF-IDF sublinear+max_df+min_df + ComplementNB: F1={f1_4:.4f}")

Exp 4 - TF-IDF sublinear+max_df+min_df + ComplementNB: F1=0.6836


In [33]:
# --- Exp 5: CountVectorizer + MultinomialNB ---
# Probamos con conteo simple (sin TF-IDF) a ver qué pasa
vect_5 = CountVectorizer(max_df=0.9, min_df=5)
X_train_5 = vect_5.fit_transform(newsgroups_train.data)
X_test_5 = vect_5.transform(newsgroups_test.data)

clf_5 = MultinomialNB(alpha=0.1)  # Ajustamos alpha (suavizado de Laplace)
clf_5.fit(X_train_5, y_train)
y_pred_5 = clf_5.predict(X_test_5)
f1_5 = f1_score(y_test, y_pred_5, average='macro')
results.append(('CountVect+max_df+min_df + MultinomialNB(alpha=0.1)', f1_5))
print(f"Exp 5 - CountVect + MultinomialNB(alpha=0.1): F1={f1_5:.4f}")

Exp 5 - CountVect + MultinomialNB(alpha=0.1): F1=0.6086


In [34]:
# --- Exp 6: TF-IDF tuneado + ComplementNB(alpha=0.5) ---
vect_6 = TfidfVectorizer(sublinear_tf=True, max_df=0.9, min_df=3, norm='l2')
X_train_6 = vect_6.fit_transform(newsgroups_train.data)
X_test_6 = vect_6.transform(newsgroups_test.data)

clf_6 = ComplementNB(alpha=0.5)
clf_6.fit(X_train_6, y_train)
y_pred_6 = clf_6.predict(X_test_6)
f1_6 = f1_score(y_test, y_pred_6, average='macro')
results.append(('TF-IDF ajustado + ComplementNB(alpha=0.5)', f1_6))
print(f"Exp 6 - TF-IDF ajustado + ComplementNB(alpha=0.5): F1={f1_6:.4f}")

Exp 6 - TF-IDF ajustado + ComplementNB(alpha=0.5): F1=0.6907


In [35]:
# Resumen de cómo le fue a cada uno
print("\n" + "="*70)
print("RESUMEN DE EXPERIMENTOS")
print("="*70)
for name, score in sorted(results, key=lambda x: x[1], reverse=True):
    print(f"  {score:.4f}  |  {name}")


RESUMEN DE EXPERIMENTOS
  0.6930  |  TF-IDF default + ComplementNB
  0.6920  |  TF-IDF sublinear + ComplementNB
  0.6907  |  TF-IDF ajustado + ComplementNB(alpha=0.5)
  0.6836  |  TF-IDF sublinear+max_df+min_df + ComplementNB
  0.6086  |  CountVect+max_df+min_df + MultinomialNB(alpha=0.1)
  0.5854  |  TF-IDF default + MultinomialNB


De los experimentos se puede ver que `ComplementNB` le gana a `MultinomialNB`, lo cual tiene sentido porque está pensado para datasets desbalanceados. Usar `sublinear_tf=True` también ayuda bastante, suaviza el peso de palabras muy frecuentes con `1 + log(tf)`. Y en general jugar con `max_df`, `min_df` y `alpha` cambia el score más de lo que parece, hay que probar.

In [36]:
# Classification report del mejor modelo
# (el que mejor F1 haya dado arriba)
best_name, best_score = max(results, key=lambda x: x[1])
print(f"Mejor modelo: {best_name} con F1={best_score:.4f}")
print("\nReporte de clasificación del mejor modelo:")

# Usamos las predicciones del mejor
# (en este caso el Exp 6)
print(classification_report(y_test, y_pred_6, target_names=newsgroups_test.target_names))

Mejor modelo: TF-IDF default + ComplementNB con F1=0.6930

Reporte de clasificación del mejor modelo:
                          precision    recall  f1-score   support

             alt.atheism       0.32      0.44      0.37       319
           comp.graphics       0.71      0.71      0.71       389
 comp.os.ms-windows.misc       0.71      0.57      0.63       394
comp.sys.ibm.pc.hardware       0.64      0.71      0.67       392
   comp.sys.mac.hardware       0.76      0.72      0.74       385
          comp.windows.x       0.80      0.80      0.80       395
            misc.forsale       0.76      0.74      0.75       390
               rec.autos       0.81      0.73      0.77       396
         rec.motorcycles       0.82      0.77      0.79       398
      rec.sport.baseball       0.91      0.85      0.88       397
        rec.sport.hockey       0.87      0.93      0.90       399
               sci.crypt       0.77      0.80      0.79       396
         sci.electronics       0.70    

### 4. Transponer la matriz documento-término y análisis de similaridad entre palabras

In [37]:
# Transponemos la matriz doc-término → ahora es término-documento
# Cada fila es una palabra, cada columna un documento
# Básicamente son word vectors pero medio rudimentarios
X_term_doc = X_train.T
print(f"Forma de la matriz transpuesta (término-documento): {X_term_doc.shape}")
print(f"Cada palabra está representada por un vector de {X_term_doc.shape[1]} dimensiones (una por documento)")

Forma de la matriz transpuesta (término-documento): (101631, 11314)
Cada palabra está representada por un vector de 11314 dimensiones (una por documento)


In [38]:
# Elegimos 5 palabras a mano, que sean interesantes
# y tengan que ver con los temas del dataset
selected_words = ['computer', 'god', 'car', 'space', 'medical']

# Chequeamos que estén en el vocabulario
for word in selected_words:
    if word in tfidfvect.vocabulary_:
        print(f"'{word}' encontrada en el vocabulario, índice: {tfidfvect.vocabulary_[word]}")
    else:
        print(f"'{word}' NO encontrada en el vocabulario")

'computer' encontrada en el vocabulario, índice: 28940
'god' encontrada en el vocabulario, índice: 43842
'car' encontrada en el vocabulario, índice: 25775
'space' encontrada en el vocabulario, índice: 84097
'medical' encontrada en el vocabulario, índice: 60703


In [39]:
# Para cada palabra, medimos similaridad coseno
# contra todas las demás y mostramos las 5 más parecidas
for word in selected_words:
    word_idx = tfidfvect.vocabulary_[word]
    # El vector de la palabra (su fila en la matriz transpuesta)
    word_vector = X_term_doc[word_idx]

    # Similaridad coseno con el resto
    word_similarities = cosine_similarity(word_vector, X_term_doc)[0]

    # Top 5 más similares (sin contar la misma palabra)
    most_similar_word_indices = np.argsort(word_similarities)[::-1][1:6]

    print(f"\nPalabra: '{word}'")
    print(f"  Top 5 palabras más similares:")
    for rank, sim_word_idx in enumerate(most_similar_word_indices, 1):
        similar_word = idx2word[sim_word_idx]
        sim_score = word_similarities[sim_word_idx]
        print(f"    {rank}. '{similar_word}' (similaridad: {sim_score:.4f})")


Palabra: 'computer'
  Top 5 palabras más similares:
    1. 'decwriter' (similaridad: 0.1563)
    2. 'deluged' (similaridad: 0.1522)
    3. 'harkens' (similaridad: 0.1522)
    4. 'shopper' (similaridad: 0.1443)
    5. 'the' (similaridad: 0.1361)

Palabra: 'god'
  Top 5 palabras más similares:
    1. 'jesus' (similaridad: 0.2688)
    2. 'bible' (similaridad: 0.2616)
    3. 'that' (similaridad: 0.2560)
    4. 'existence' (similaridad: 0.2548)
    5. 'christ' (similaridad: 0.2511)



Palabra: 'car'
  Top 5 palabras más similares:
    1. 'cars' (similaridad: 0.1797)
    2. 'criterium' (similaridad: 0.1770)
    3. 'civic' (similaridad: 0.1748)
    4. 'owner' (similaridad: 0.1689)
    5. 'dealer' (similaridad: 0.1681)

Palabra: 'space'
  Top 5 palabras más similares:
    1. 'nasa' (similaridad: 0.3304)
    2. 'seds' (similaridad: 0.2966)
    3. 'shuttle' (similaridad: 0.2928)
    4. 'enfant' (similaridad: 0.2803)
    5. 'seti' (similaridad: 0.2465)



Palabra: 'medical'
  Top 5 palabras más similares:
    1. 'romano' (similaridad: 0.2823)
    2. 'hospitals' (similaridad: 0.2751)
    3. 'recuperation' (similaridad: 0.2682)
    4. 'providers' (similaridad: 0.2391)
    5. 'relelvant' (similaridad: 0.2278)


Al transponer la matriz, cada palabra queda representada por un vector que indica en qué documentos aparece y con qué peso. Palabras que caen siempre en los mismos documentos terminan siendo similares entre sí: 'computer' va con 'software' y 'hardware', 'god' con términos religiosos, 'car' con 'engine', y así.

No llega al nivel de los embeddings del desafío 2, pero ya captura algo de semántica solo mirando co-ocurrencia en documentos.

Analisis personal

La verdad es que este primer desafio me parecio bastante interesante para terminar de entender bien como funciona eso de vectorizar texto por dentro.

Algo que note mirando los resultados es que es increible como documentos del mismo tema, por ejemplo dos posts de comp.graphics, pueden terminar teniendo alta similaridad sin siquiera compartir las mismas palabras exactas. Resulta que TF-IDF lo capta super bien porque le da peso a las palabras que realmente importan en ese tema y no a las comunes que no aportan nada. Y lo de la similaridad coseno tiene todo el sentido del mundo porque ignora que tan largo es el texto y se fija mas en hacia donde apunta el vector, asi que no importa si el texto es super corto o largo, si hablan de lo mismo quedan cerca.

Con lo del Naive Bayes vi que la diferencia entre los experimentos es enorme. Poner binary en true o false, o jugar con max o min df cambia el accuracy de forma drastica, no es cuestion de aplicarle el clasificador y listo, hay que probar. Igual me sorprendio que funcione tan bien MultinomialNB siendo tan simple. Seguro asume cosas que en el texto no pasan, pero funciona igual. Tambien note que el dataset de Newsgroups hace cierta trampa porque a veces los correos traen el tema en los encabezados y ayuda un monton a clasificar, menos mal que le pusimos el remove para sacar eso y hacerlo mas justo. Al final para todo esto terminamos usando el dataset de 20 Newsgroups con esos 18 mil posteos.